# Análisis de métricas financieras — criptomonedas

Con el dataset unificado de `precios_diarios.csv` como input, este notebook calcula las cuatro métricas que permiten comparar el perfil de riesgo de cada criptomoneda: retorno diario, volatilidad, drawdown y ratio riesgo/retorno.

Output: `metricas_criptos.csv` — tabla de 5 filas con el perfil de riesgo de cada activo.

In [1]:
# 02_analisis_metricas.ipynb
# Análisis financiero de criptomonedas
# Calcula métricas clave y genera CSV listo para dashboards

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

## 1. Configuración de rutas

In [2]:
# -----------------------------
# 1. Configuración de rutas
# -----------------------------
ruta_csv = "../datos/procesados/precios_diarios.csv"
ruta_metricas = "../datos/procesados/metricas_criptos.csv"

## 2. Carga del dataset unificado

In [ ]:
# -----------------------------
# 2. Cargar dataset unificado
# -----------------------------
print(f"Leyendo dataset desde: {ruta_csv}")
df = pd.read_csv(ruta_csv, encoding="utf-8", sep=';')
print(f"  Filas leídas: {len(df)}, columnas: {list(df.columns)}")

df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True)
print("  Columna 'fecha' convertida a datetime")

# Ordenar por cripto y fecha
df = df.sort_values(["cripto_id", "fecha"]).reset_index(drop=True)
print("  Dataset ordenado por cripto_id y fecha")

# Revisar criptos incluidas
print("Criptomonedas en el dataset:", df["cripto_id"].unique())
print("Rango de fechas:", df["fecha"].min().date(), "->", df["fecha"].max().date())

## 3. Retorno diario

Se agrupa por `cripto_id` **antes** de aplicar `pct_change()`. Sin el `groupby`, el primer registro de cada criptomoneda compararía su precio con el último precio de la cripto anterior en el DataFrame concatenado, produciendo un retorno ficticio en esa fila.

In [ ]:
# -----------------------------
# 3. Calcular retorno diario
# -----------------------------
df["retorno_diario"] = df.groupby("cripto_id")["cierre"].pct_change()
nulos = df["retorno_diario"].isna().sum()
print(f"Retorno diario calculado. Valores nulos (primer día de cada cripto): {nulos} de {len(df)} filas")
print(df[["cripto_id","fecha","cierre","retorno_diario"]].head(3))

## 4. Drawdown

Mide cuánto ha caído el precio desde el máximo histórico acumulado hasta esa fecha. `cummax()` es progresivo — nunca decrece — garantizando que el drawdown sea siempre ≥ 0. Captura la pérdida real del inversor: cuánto ha bajado desde el mejor momento en que podría haber salido.

In [ ]:
# -----------------------------
# 4. Calcular drawdown diario
# -----------------------------
df["cierre_max"] = df.groupby("cripto_id")["cierre"].cummax()
df["drawdown"] = df["cierre_max"] - df["cierre"]
print("Drawdown calculado para cada fila.")
print(f"  Drawdown mínimo (debe ser 0, en los picos): {df['drawdown'].min()}")
print(f"  Drawdown máximo global: {df['drawdown'].max():.2f}")

## 5. Métricas agregadas por criptomoneda

Las cinco métricas se calculan en una sola pasada con `groupby().apply()`. La elección responde a un criterio de complementariedad:

- **Retorno promedio** y **volatilidad** por separado son insuficientes: un activo puede tener retorno alto con riesgo desproporcionado.
- El **ratio riesgo/retorno** (retorno medio / volatilidad) combina ambas en una dimensión comparable, análogo al ratio de Sharpe sin tasa libre de riesgo.
- El **drawdown máximo** añade la dimensión de pérdida extrema que el ratio no captura: dos activos con el mismo ratio pueden diferir mucho en la caída máxima que exigen soportar.

In [ ]:
# -----------------------------
# 5. Función para métricas por cripto
# -----------------------------
def calcular_metricas(grupo):
    cripto = grupo["cripto_id"].iloc[0]
    retorno = grupo["retorno_diario"].dropna()
    drawdown_max = grupo["drawdown"].max()
    print(f"  Calculando métricas para {cripto}: {len(grupo)} registros, {len(retorno)} retornos válidos")
    return pd.Series({
        "precio_medio": grupo["cierre"].mean(),
        "volatilidad": retorno.std(),
        "retorno_promedio": retorno.mean(),
        "drawdown_max": drawdown_max,
        "ratio_riesgo_retorno": retorno.mean() / retorno.std() if retorno.std() != 0 else np.nan
    })

# Aplicar por cripto
print("Calculando métricas agregadas por criptomoneda...")
metricas = df.groupby("cripto_id").apply(calcular_metricas).reset_index()
print(f"\nMétricas calculadas para {len(metricas)} criptomonedas")

## 6. Persistencia de métricas

In [ ]:
# -----------------------------
# 6. Guardar métricas en CSV
# -----------------------------
print(f"Guardando métricas en: {ruta_metricas}")
metricas.to_csv(ruta_metricas, index=False, encoding="utf-8", sep=';')
print("Métricas guardadas en:", ruta_metricas)
print(f"\nTabla final de métricas ({len(metricas)} filas):")
print(metricas)

## 7. Visualizaciones

Tres gráficos cubren dimensiones independientes del riesgo:

1. **Precio histórico** — confirma el rango temporal y los órdenes de magnitud. Permite detectar si algún CSV tiene datos anómalos.
2. **Retorno acumulado** — normaliza el efecto del precio absoluto. Muestra cuánto habría crecido una unidad monetaria invertida al inicio de cada serie.
3. **Drawdown** — visualiza en qué periodos cada activo estuvo más alejado de su pico histórico, clave para evaluar la resistencia psicológica que exigiría mantener la posición.

In [ ]:
# -----------------------------
# 7. Gráficos rápidos (opcional)
# -----------------------------
# Precio de cierre histórico
print("Generando gráfico 1/3: precio de cierre histórico...")
plt.figure(figsize=(12,6))
for cripto in df["cripto_id"].unique():
    subset = df[df["cripto_id"] == cripto]
    plt.plot(subset["fecha"], subset["cierre"], label=cripto)

plt.title("Precio de cierre histórico por criptomoneda")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.legend()
plt.show()

# Retorno diario acumulado
print("Generando gráfico 2/3: retorno acumulado...")
plt.figure(figsize=(12,6))
for cripto in df["cripto_id"].unique():
    subset = df[df["cripto_id"] == cripto].copy()
    subset["retorno_acum"] = (1 + subset["retorno_diario"].fillna(0)).cumprod()
    plt.plot(subset["fecha"], subset["retorno_acum"], label=cripto)

plt.title("Retorno acumulado histórico")
plt.xlabel("Fecha")
plt.ylabel("Retorno acumulado")
plt.legend()
plt.show()

# Drawdown máximo por cripto
print("Generando gráfico 3/3: drawdown histórico...")
plt.figure(figsize=(12,6))
for cripto in df["cripto_id"].unique():
    subset = df[df["cripto_id"] == cripto]
    plt.plot(subset["fecha"], subset["drawdown"], label=cripto)

plt.title("Drawdown histórico por criptomoneda")
plt.xlabel("Fecha")
plt.ylabel("Drawdown")
plt.legend()
plt.show()

print("Gráficos generados correctamente.")